In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from pathlib import Path

data_path = Path('nearest-earth-objects(1910-2024).csv')
if not data_path.is_file():
    raise FileNotFoundError(
        f"Dataset not found: {data_path.resolve()}. Download it from the Kaggle source linked in the README and save it in the repository root."
    )
df = pd.read_csv(data_path)

In [ ]:
df.head()

In [ ]:
df.drop(['neo_id','name','orbiting_body'], axis=1, inplace=True)

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)
df

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
def fill_null_value(df):
  for col in df.columns:
    if df[col].dtype == 'object':
      df[col].fillna(df[col].mode()[0], inplace=True)
    else:
      df[col].fillna(df[col].median(), inplace=True)

  return df

In [ ]:
df = fill_null_value(df)

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print(df)

In [ ]:
for col in df.columns:
    print( col,':', df[col].nunique())
    print(df[col].value_counts().nlargest(5))
    print('\n' + '*' * 20 + '\n')

In [ ]:
plt.figure(figsize=(12, 6))
plt.hist(df['absolute_magnitude'], bins=30, color='g', edgecolor='k', alpha=0.7)
plt.title('Distribution of Absolute Magnitude')
plt.xlabel('Absolute Magnitude')
plt.ylabel('Frequency')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.scatter(df['estimated_diameter_min'], df['estimated_diameter_max'], alpha=0.5, color='r')
plt.title('Estimated Diameter Min vs Estimated Diameter Max')
plt.xlabel('Estimated Diameter Min')
plt.ylabel('Estimated Diameter Max')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x=df['is_hazardous'], y=df['relative_velocity'])
plt.title('Relative Velocity by Hazardous Status')
plt.xlabel('Is Hazardous?')
plt.ylabel('Relative Velocity')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

In [ ]:
X

In [ ]:
y

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Applying LogisticRegression Model 

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
LogisticRegressionModel = LogisticRegression(solver='sag',C=1,random_state=33,max_iter=200)

In [ ]:
LogisticRegressionModel.fit(X_train_scaled, y_train)

In [ ]:
y_pred = LogisticRegressionModel.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, zero_one_loss 

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", conf_matrix)

In [ ]:
print('LogisticRegressionModel Train Score is: ', LogisticRegressionModel.score(X_train_scaled, y_train))
print('LogisticRegressionModel Test Score is: ', LogisticRegressionModel.score(X_test_scaled, y_test))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
photo = sns.heatmap(cm, annot=True, cmap='Blues', fmt='g')
photo.set_xlabel('Predicted labels')
photo.set_ylabel('True labels')
photo.set_title('Confusion Matrix of Logistic Regression Model')
plt.show()
photo.figure.savefig('logistic_regression_confusion_matrix.png')

In [ ]:
AccScore = accuracy_score(y_test, y_pred) 
print('Accuracy Score is: ', AccScore)

In [ ]:
ZeroOneLossValue = zero_one_loss(y_test, y_pred, normalize=False) 
print('Zero One Loss Value: ', ZeroOneLossValue)

# Applying Decision Tree Model

In [ ]:
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier(max_depth=5, min_samples_leaf=10)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
y_pred_prob =model .predict_proba(X_test)

In [ ]:
y_pred =model .predict(X_test) 
cm = confusion_matrix(y_test, y_pred)
print(cm)

In [ ]:
photo = sns.heatmap(cm, annot=True, cmap='Blues', fmt='g')
photo.set_xlabel('Predicted labels')
photo.set_ylabel('True labels')
photo.set_title('Confusion Matrix of Decision Tree Model')
plt.show()
photo.figure.savefig('decision_tree_confusion_matrix.png')

In [ ]:
print('Decision Tree Model Train Score is: ' ,model.score(X_train, y_train))
print('Decision Tree Model Test Score is: ' , model.score(X_test, y_test))

In [ ]:
results = accuracy_score(y_test, y_pred) 
print('Accuracy Score is: ', results)

In [ ]:
ZeroOneLossValue = zero_one_loss(y_test,y_pred,normalize=False) 
print('Zero One Loss Value: ', ZeroOneLossValue)

# Applying Random Forest  Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.metrics import balanced_accuracy_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import roc_auc_score, roc_curve

In [ ]:
# Decision-tree baseline; the Random Forest is evaluated below.
balance_acc = balanced_accuracy_score(y_test, y_pred)
print('Decision Tree Balanced Accuracy:', balance_acc)

In [ ]:
model = RandomForestClassifier(class_weight='balanced')
model.fit(X_train, y_train)

In [ ]:
undersampler = RandomUnderSampler()
X_resampled, y_resampled = undersampler.fit_resample(X, y)

In [ ]:
X = df.drop(columns=['is_hazardous'])
y = df['is_hazardous']
smote = SMOTE()
X_resampled, y_resampled = smote.fit_resample(X, y)

In [ ]:
target_column = 'is_hazardous'

In [ ]:
X = df.drop(columns=[target_column])
y = df[target_column]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

In [ ]:
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

In [ ]:
print(classification_report(y_test, y_pred_rf))

In [ ]:
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

auc_roc = roc_auc_score(y_test, y_prob_rf)
print("AUC-ROC:", auc_roc)

fpr, tpr, thresholds = roc_curve(y_test, y_prob_rf)
plt.plot(fpr, tpr, label=f"AUC-ROC = {auc_roc:.2f}")
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='best')
plt.show()

In [ ]:
print('Random Forest model accuracy on test data:', accuracy_score(y_test, y_pred_rf))
print('Classification report for the Random Forest model:\n', classification_report(y_test, y_pred_rf))

In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf)
print(cm_rf)

In [ ]:
plt.figure(figsize=(8, 6))
photo = sns.heatmap(cm_rf, annot=True, cmap='Blues', fmt='g', xticklabels=rf_model.classes_, yticklabels=rf_model.classes_)
photo.set_xlabel('Predicted labels')
photo.set_ylabel('True labels')
photo.set_title('Confusion Matrix of Random Forest Model')
plt.show()
photo.figure.savefig('random_forest_confusion_matrix.png')

In [ ]:
ran = accuracy_score(y_test, y_pred_rf) 
print('Accuracy Score is: ', ran)

In [ ]:
ZeroOneLossnow = zero_one_loss(y_test, y_pred_rf, normalize=False)
print('Zero One Loss Value: ', ZeroOneLossnow)

In [ ]:
print("Random Forest Accuracy: ", ran )   
print("Logistic Regression Accuracy: ", AccScore )
print("Decision Tree Accuracy:", results )